# **Exploracion de movilidad mensual del Lago de Atitlan**

Grupo 3 - CC3084 Data Science.


## **1. Preparacion de datos**

### **1.1. Carga de la fuente**

El archivo `BD_moviidad_mensual.xlsx` contiene informacion mensual sobre la afluencia de visitantes a distintos destinos turisticos de Guatemala. Para este analisis se carga la hoja `Atitlan`, correspondiente a los municipios del Lago de Atitlan.


In [1]:
from pathlib import Path
import pandas as pd

raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "data" / "raw").is_dir():
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / "data" / "raw" / "BD_moviidad_mensual.xlsx"
datos = pd.read_excel(
    ruta_datos,
    sheet_name="Atitlan",
    dtype=str,
    keep_default_na=False,
)


### **1.2. Primeras cinco filas**

A continuacion, mostramos las primeras cinco filas para comenzar a reconocer la estructura de los registros y las variables disponibles.


In [2]:
datos.head()


,destino,sub_grupo,anio,mes,pais,es_consecutivo,fecha,visitantes,dias_persona,permanencia_prom
0,Panajachel,Lago Atitlan,2025,9,Alemania,mayor a 1 dia,2025-09-01 00:00:00,20,51,2.55
1,Panajachel,Lago Atitlan,2025,9,Alemania,menor a 1 dia,2025-09-01 00:00:00,28,28,1
2,Panajachel,Lago Atitlan,2025,9,Arabia Saudita,menor a 1 dia,2025-09-01 00:00:00,2,2,1
3,Panajachel,Lago Atitlan,2025,9,Argentina,mayor a 1 dia,2025-09-01 00:00:00,5,12,2.4
4,Panajachel,Lago Atitlan,2025,9,Argentina,menor a 1 dia,2025-09-01 00:00:00,14,14,1


## **2. Descripcion de las variables**

- **destino:** municipio del Lago de Atitlan al que corresponde el registro de visitantes, por lo que su tipo es cualitativa nominal.

- **sub_grupo:** indica la zona turistica a la que pertenece el lugar del registro, por lo que su tipo es cualitativa nominal. En las 9,860 filas de esta hoja tiene el mismo valor, `Lago Atitlan`. Como no permite diferenciar los registros de este analisis, mas adelante se podria considerar descartar esta columna.

- **anio:** anio al que corresponde el registro de visitantes; variable temporal de tipo cuantitativa discreta.

- **mes:** numero del mes al que corresponde el registro; variable cualitativa ordinal de caracter ciclico.

- **pais:** pais de residencia de los visitantes incluidos en el registro; variable cualitativa nominal.

- **es_consecutivo:** clasifica a los visitantes segun su permanencia en dos grupos: menor a un dia y mayor a un dia, por lo que su tipo es cualitativa ordinal. Para un mismo mes, municipio y pais de procedencia puede haber un registro de visitantes que permanecieron menos de un dia y otro de quienes permanecieron mas de un dia.

- **fecha:** contiene el anio, mes, dia y hora del periodo registrado. En todas las filas la hora es `00:00:00`; variable temporal de fecha y hora.

- **visitantes:** cantidad de visitantes reportada para un mismo mes, municipio, pais de procedencia y grupo de permanencia. Aunque conceptualmente representa un conteo de personas, la fuente incluye valores con decimales, por lo que en su representacion actual se tratara como cuantitativa continua. El procedimiento que origina esos decimales no esta completamente documentado. Para una presentacion mas directa, se plantea mostrar el resultado final aproximado al entero superior; los calculos conservaran los valores originales y el redondeo se aplicara unicamente al presentar los resultados.

- **dias_persona:** suma de los dias de permanencia de todos los visitantes incluidos en el registro. Por ejemplo, si una persona permanece tres dias y otra dos, entre ambas acumulan cinco dias-persona. Conceptualmente es una cantidad acumulada en dias-persona; como la fuente incluye cifras con decimales, en su representacion actual se tratara como cuantitativa continua.

- **permanencia_prom:** cantidad promedio de dias que permanece cada visitante del registro; variable cuantitativa continua. Es un valor derivado que se obtiene al dividir los dias-persona entre la cantidad de visitantes. Por ejemplo, 51 dias-persona entre 20 visitantes equivalen a 2.55 dias por visitante. Como puede recalcularse a partir de esas dos variables, mas adelante se podria considerar descartar esta columna y calcularla cuando se necesite. Para obtener la permanencia de varios registros, se dividen sus dias-persona acumulados entre sus visitantes acumulados, utilizando solo el grupo de permanencia mayor a un dia; no se promedian directamente los valores de esta columna.


## **3. Revision de calidad de los datos**

Se revisa la hoja completa para identificar faltantes, duplicados, problemas de formato y posibles contradicciones entre las variables. Las conversiones siguientes se utilizan solamente para el diagnostico: `datos` conserva los valores originales y no se guarda un dataset limpio.

### **3.1. Estructura, categorias y formato del archivo**

Primero, se comprueba el tamano de la hoja, las categorias disponibles y si existen columnas constantes. Los tipos de pandas aparecen como texto porque asi se cargo el archivo; esto no sustituye la clasificacion conceptual que se hizo para cada variable.

In [3]:
import math
import unicodedata
from openpyxl import load_workbook

datos_originales = datos.copy(deep=True)
marcadores_nulo = {"null", "nan", "none", "n/a", "na"}
texto = datos.astype("string")
vacios = texto.isna() | texto.apply(lambda columna: columna.str.strip().eq(""))
marcadores = texto.apply(lambda columna: columna.str.strip().str.lower().isin(marcadores_nulo))
faltantes = vacios | marcadores

def ejemplos(mascara, columnas, limite=10):
    resultado = datos.loc[mascara.fillna(False), columnas].copy()
    resultado.insert(0, "fila_excel", resultado.index + 2)
    return resultado.head(limite)

estructura = pd.DataFrame({
    "variable": datos.columns,
    "tipo_pandas": [str(datos[columna].dtype) for columna in datos],
    "valores_distintos_originales": [datos[columna].nunique(dropna=False) for columna in datos],
})
estructura["constante"] = estructura["valores_distintos_originales"].eq(1)
print(f"Se revisan {len(datos):,} filas y {len(datos.columns)} columnas.")
estructura

Se revisan 9,860 filas y 10 columnas.


,variable,tipo_pandas,valores_distintos_originales,constante
0,destino,object,10,False
1,sub_grupo,object,1,True
2,anio,object,2,False
3,mes,object,10,False
4,pais,object,101,False
5,es_consecutivo,object,2,False
6,fecha,object,10,False
7,visitantes,object,1584,False
8,dias_persona,object,1772,False
9,permanencia_prom,object,1483,False


In [4]:
columnas_categoricas = ["destino", "sub_grupo", "anio", "mes", "es_consecutivo"]
categorias = pd.DataFrame([
    {"variable": columna, "valor": valor, "registros": cantidad}
    for columna in columnas_categoricas
    for valor, cantidad in datos[columna].value_counts(dropna=False).items()
])
categorias

,variable,valor,registros
0,destino,Panajachel,1302
1,destino,San Pedro La Laguna,1265
2,destino,Santiago Atitlan,1126
3,destino,San Juan La Laguna,1069
4,destino,San Marcos La Laguna,1009
5,destino,Santa Catarina Palopo,929
6,destino,San Pablo La Laguna,878
7,destino,San Antonio Palopo,843
8,destino,San Lucas Toliman,825
9,destino,Santa Cruz La Laguna,614


In [5]:
libro = load_workbook(ruta_datos, read_only=True, data_only=False)
try:
    hoja = libro["Atitlan"]
    dimension_declarada = hoja.calculate_dimension()
    hoja.reset_dimensions()
    errores_excel, formulas_excel = [], []
    filas_fisicas, ancho_real = 0, 0
    for fila in hoja.iter_rows():
        filas_fisicas += 1
        ancho_real = max(ancho_real, len(fila))
        for celda in fila:
            if celda.data_type == "e":
                errores_excel.append((celda.coordinate, celda.value))
            elif celda.data_type == "f":
                formulas_excel.append((celda.coordinate, celda.value))
finally:
    libro.close()

revision_excel = pd.DataFrame([
    {"revision": "Dimension declarada por Excel", "resultado": dimension_declarada},
    {"revision": "Filas reales incluyendo cabecera", "resultado": filas_fisicas},
    {"revision": "Columnas reales", "resultado": ancho_real},
    {"revision": "Lectura de todas las filas", "resultado": len(datos) == filas_fisicas - 1},
    {"revision": "Celdas con error de Excel", "resultado": len(errores_excel)},
    {"revision": "Celdas con formulas de Excel", "resultado": len(formulas_excel)},
])
revision_excel

,revision,resultado
0,Dimension declarada por Excel,A1:A1
1,Filas reales incluyendo cabecera,9861
2,Columnas reales,10
3,Lectura de todas las filas,True
4,Celdas con error de Excel,0
5,Celdas con formulas de Excel,0


La dimension declarada del libro no describe su contenido real: indica una sola celda, aunque existen miles de registros. La lectura utilizada si recupera la hoja completa. Las celdas vacias se investigan por separado de los errores de Excel y de las formulas.

### **3.2. Valores faltantes y marcadores escritos como texto**

Una revision basada solo en `isna()` no detectaria las cadenas vacias ni el texto `Null`, porque conservamos esos valores durante la carga. Por eso se cuentan por separado los nulos reconocidos por pandas, los textos vacios y los marcadores de ausencia. Los porcentajes siguientes corresponden a registros de la tabla, no a porcentajes de visitantes.

In [6]:
resumen_faltantes = pd.DataFrame({
    "nulos_pandas": datos.isna().sum(),
    "vacios_texto": texto.apply(lambda columna: columna.str.strip().eq("").fillna(False).sum()),
    "marcadores_texto": marcadores.sum(),
    "total_sin_dato": faltantes.sum(),
})
resumen_faltantes["porcentaje_registros"] = (resumen_faltantes["total_sin_dato"] / len(datos) * 100).round(3)
resumen_faltantes

,nulos_pandas,vacios_texto,marcadores_texto,total_sin_dato,porcentaje_registros
destino,0,0,0,0,0.0
sub_grupo,0,0,0,0,0.0
anio,0,0,0,0,0.0
mes,0,0,0,0,0.0
pais,0,13,12,25,0.254
es_consecutivo,0,0,0,0,0.0
fecha,0,0,0,0,0.0
visitantes,0,0,0,0,0.0
dias_persona,0,0,0,0,0.0
permanencia_prom,0,13,0,13,0.132


In [7]:
columnas_contexto = ["destino", "fecha", "pais", "es_consecutivo", "visitantes", "dias_persona", "permanencia_prom"]
casos_pais_faltante = ejemplos(faltantes["pais"], columnas_contexto, limite=len(datos))
casos_pais_faltante

,fila_excel,destino,fecha,pais,es_consecutivo,visitantes,dias_persona,permanencia_prom
324,326,Panajachel,2025-11-01 00:00:00,Null,menor a 1 dia,2,2,1
606,608,Panajachel,2026-03-01 00:00:00,Null,menor a 1 dia,1,1,1
1172,1174,Panajachel,2026-08-01 00:00:00,,mayor a 1 dia,2,6,3
1173,1175,Panajachel,2026-08-01 00:00:00,,menor a 1 dia,2,2,1
1496,1498,San Antonio Palopo,2025-11-01 00:00:00,Null,menor a 1 dia,1,1,1
2415,2417,San Juan La Laguna,2025-11-01 00:00:00,Null,menor a 1 dia,1,1,1
2646,2648,San Juan La Laguna,2026-03-01 00:00:00,Null,mayor a 1 dia,1,1,1
3102,3104,San Juan La Laguna,2026-08-01 00:00:00,,mayor a 1 dia,2,4,2
3103,3105,San Juan La Laguna,2026-08-01 00:00:00,,menor a 1 dia,2,2,1
4508,4510,San Marcos La Laguna,2026-03-01 00:00:00,Null,mayor a 1 dia,1,1,1


In [8]:
casos_permanencia_faltante = ejemplos(faltantes["permanencia_prom"], columnas_contexto, limite=len(datos))
casos_permanencia_faltante

,fila_excel,destino,fecha,pais,es_consecutivo,visitantes,dias_persona,permanencia_prom
601,603,Panajachel,2026-03-01 00:00:00,Nigeria,menor a 1 dia,0,0,
790,792,Panajachel,2026-05-01 00:00:00,Arabia Saudita,menor a 1 dia,0,0,
3310,3312,San Lucas Toliman,2025-10-01 00:00:00,Estonia,menor a 1 dia,0,0,
3567,3569,San Lucas Toliman,2026-03-01 00:00:00,Estonia,menor a 1 dia,0,0,
6213,6215,San Pedro La Laguna,2025-11-01 00:00:00,Indonesia,menor a 1 dia,0,0,
6278,6280,San Pedro La Laguna,2025-11-01 00:00:00,Ucrania,menor a 1 dia,0,0,
6352,6354,San Pedro La Laguna,2025-12-01 00:00:00,Indonesia,menor a 1 dia,0,0,
6365,6367,San Pedro La Laguna,2025-12-01 00:00:00,Letonia,menor a 1 dia,0,0,
6941,6943,San Pedro La Laguna,2026-07-01 00:00:00,Arabia Saudita,menor a 1 dia,0,0,
7180,7182,San Pedro La Laguna,2026-08-01 00:00:00,Tailandia,menor a 1 dia,0,0,


### **3.3. Duplicados y variantes de escritura**

La clave esperada de un registro es municipio, mes, pais y grupo de permanencia. Se buscan filas repetidas y claves repetidas. Tambien se comprueba si espacios, mayusculas o tildes ocultan categorias equivalentes. La normalizacion se aplica en una copia de las claves, sin corregir la fuente. No se consideran duplicados dos registros del mismo municipio y mes si corresponden a paises o grupos de permanencia diferentes.

In [9]:
clave = ["destino", "fecha", "pais", "es_consecutivo"]

def normalizar_etiqueta(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).strip().casefold()
    if valor in marcadores_nulo:
        return ""
    return "".join(caracter for caracter in unicodedata.normalize("NFKD", valor) if not unicodedata.combining(caracter))

claves_normalizadas = datos[clave].apply(lambda columna: columna.map(normalizar_etiqueta))
duplicados_exactos = datos.duplicated(keep=False)
duplicados_clave = datos.duplicated(clave, keep=False)
duplicados_normalizados = claves_normalizadas.duplicated(keep=False)
revision_duplicados = pd.DataFrame([
    {"revision": "Filas involucradas en duplicados exactos", "registros": int(duplicados_exactos.sum())},
    {"revision": "Filas involucradas en claves repetidas", "registros": int(duplicados_clave.sum())},
    {"revision": "Filas involucradas en claves normalizadas repetidas", "registros": int(duplicados_normalizados.sum())},
    {"revision": "Filas completamente vacias", "registros": int(vacios.all(axis=1).sum())},
])
revision_duplicados

,revision,registros
0,Filas involucradas en duplicados exactos,0
1,Filas involucradas en claves repetidas,0
2,Filas involucradas en claves normalizadas repe...,0
3,Filas completamente vacias,0


In [10]:
variantes = []
for columna in ["destino", "sub_grupo", "pais", "es_consecutivo"]:
    presentes = datos.loc[~faltantes[columna], columna]
    tabla = pd.DataFrame({"original": presentes, "normalizado": presentes.map(normalizar_etiqueta)})
    for etiqueta, grupo in tabla.groupby("normalizado"):
        valores = sorted(grupo["original"].unique())
        if len(valores) > 1:
            variantes.append({"variable": columna, "categoria": etiqueta, "variantes": valores})

revision_texto = pd.DataFrame({
    "espacios_exteriores": texto.apply(lambda columna: columna.ne(columna.str.strip()).fillna(False).sum()),
    "posible_codificacion_danada": texto.apply(lambda columna: columna.str.contains(r"[\u00c3\u00c2\ufffd]", regex=True, na=False).sum()),
})
print(f"Grupos con variantes equivalentes: {len(variantes)}")
print("Esta busqueda no valida todos los paises contra un catalogo oficial.")
revision_texto

Grupos con variantes equivalentes: 0
Esta busqueda no valida todos los paises contra un catalogo oficial.


,espacios_exteriores,posible_codificacion_danada
destino,0,0
sub_grupo,0,0
anio,0,0
mes,0,0
pais,0,0
es_consecutivo,0,0
fecha,0,0
visitantes,0,0
dias_persona,0,0
permanencia_prom,0,0


### **3.4. Conversion numerica, rangos y decimales**

Se convierten las columnas numericas en una vista de diagnostico para detectar valores que no se puedan interpretar, negativos o no finitos. Que una celda se convierta correctamente no demuestra por si solo que su valor sea correcto. Los decimales se conservan y no se redondean en esta revision.

In [11]:
columnas_numericas = ["anio", "mes", "visitantes", "dias_persona", "permanencia_prom"]
numericos = datos[columnas_numericas].apply(pd.to_numeric, errors="coerce")
revision_numerica = []
for columna in columnas_numericas:
    serie = numericos[columna]
    no_finitos = serie.notna() & ~serie.map(lambda valor: math.isfinite(valor) if pd.notna(valor) else True)
    decimales = serie.notna() & serie.sub(serie.round()).abs().gt(1e-8)
    revision_numerica.append({
        "variable": columna,
        "texto_no_convertible": int((~faltantes[columna] & serie.isna()).sum()),
        "no_finitos": int(no_finitos.sum()),
        "negativos": int(serie.lt(0).sum()),
        "ceros": int(serie.eq(0).sum()),
        "registros_con_decimales": int(decimales.sum()),
        "minimo": serie.min(), "maximo": serie.max(),
    })
revision_numerica = pd.DataFrame(revision_numerica)
revision_numerica

,variable,texto_no_convertible,no_finitos,negativos,ceros,registros_con_decimales,minimo,maximo
0,anio,0,0,0,0,0,2025.0,2026.000000
1,mes,0,0,0,0,0,3.0,12.000000
2,visitantes,0,0,0,13,998,0.0,152399.452194
3,dias_persona,0,0,0,13,998,0.0,498014.261987
4,permanencia_prom,0,0,0,0,2925,1.0,30.000000


In [12]:
con_decimales = numericos["visitantes"].sub(numericos["visitantes"].round()).abs().gt(1e-8)
decimales_por_pais = (
    datos.loc[con_decimales].groupby("pais").size().rename("registros_con_decimales").to_frame()
)
decimales_por_pais

,registros_con_decimales
pais,
Costa Rica,200
El Salvador,200
Guatemala,200
Honduras,199
Nicaragua,199


In [13]:
ejemplos_decimales = ejemplos(con_decimales, columnas_contexto, limite=5)
ejemplos_decimales

,fila_excel,destino,fecha,pais,es_consecutivo,visitantes,dias_persona,permanencia_prom
25,27,Panajachel,2025-09-01 00:00:00,Costa Rica,mayor a 1 dia,634.9206550633196,1202.2965595879873,1.8936170212765944
26,28,Panajachel,2025-09-01 00:00:00,Costa Rica,menor a 1 dia,1857.4806398128867,1857.4806398128867,1
32,34,Panajachel,2025-09-01 00:00:00,El Salvador,mayor a 1 dia,503.7179071921817,985.8479040761271,1.957142857142857
33,35,Panajachel,2025-09-01 00:00:00,El Salvador,menor a 1 dia,954.6653669642274,954.6653669642274,1
46,48,Panajachel,2025-09-01 00:00:00,Guatemala,mayor a 1 dia,69687.71664037088,400612.9989498473,5.748688840204697


Los decimales tambien aparecen en mercados distintos de Guatemala. Este hallazgo no demuestra que sean promedios ni permite deducir por si solo como se expandio cada mercado. Se conservan las cifras originales; su presentacion y la metodologia de estimacion son asuntos diferentes.

### **3.5. Fechas y cobertura mensual**

Se comprueba que las fechas sean interpretables, que coincidan con anio y mes y que representen el primer dia del mes a medianoche. Luego se revisa la cobertura de cada municipio. Un conteo de cero registros indica ausencia de filas, no cero visitantes; no se completan las medidas de los meses ausentes.

In [14]:
fechas = pd.to_datetime(datos["fecha"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
revision_fechas = pd.DataFrame([
    {"revision": "Fechas no interpretables y no vacias", "registros": int((~faltantes["fecha"] & fechas.isna()).sum())},
    {"revision": "Anio no entero", "registros": int(numericos["anio"].sub(numericos["anio"].round()).abs().gt(1e-8).sum())},
    {"revision": "Mes no entero", "registros": int(numericos["mes"].sub(numericos["mes"].round()).abs().gt(1e-8).sum())},
    {"revision": "Mes fuera de 1 a 12", "registros": int((numericos["mes"].notna() & ~numericos["mes"].between(1, 12)).sum())},
    {"revision": "Anio no coincide con fecha", "registros": int((fechas.notna() & numericos["anio"].notna() & fechas.dt.year.ne(numericos["anio"])).sum())},
    {"revision": "Mes no coincide con fecha", "registros": int((fechas.notna() & numericos["mes"].notna() & fechas.dt.month.ne(numericos["mes"])).sum())},
    {"revision": "Dia distinto del primero del mes", "registros": int((fechas.notna() & fechas.dt.day.ne(1)).sum())},
    {"revision": "Hora distinta de medianoche", "registros": int((fechas.notna() & fechas.ne(fechas.dt.normalize())).sum())},
])
revision_fechas

,revision,registros
0,Fechas no interpretables y no vacias,0
1,Anio no entero,0
2,Mes no entero,0
3,Mes fuera de 1 a 12,0
4,Anio no coincide con fecha,0
5,Mes no coincide con fecha,0
6,Dia distinto del primero del mes,0
7,Hora distinta de medianoche,0


In [15]:
periodos_ventana = pd.period_range("2025-09", "2026-08", freq="M")
cobertura = pd.crosstab(datos["destino"], fechas.dt.to_period("M")).reindex(columns=periodos_ventana, fill_value=0)
meses_ausentes = [str(periodo) for periodo in cobertura.columns[cobertura.sum(axis=0).eq(0)]]
print("Meses sin registros en la ventana de trabajo:", meses_ausentes)
print("La tabla cuenta filas disponibles por municipio y mes, no visitantes.")
cobertura

Meses sin registros en la ventana de trabajo: ['2026-01', '2026-02']
La tabla cuenta filas disponibles por municipio y mes, no visitantes.


,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
destino,,,,,,,,,,,,
Panajachel,116,118,131,138,0,0,145,139,136,122,127,130
San Antonio Palopo,65,69,85,81,0,0,93,98,87,78,93,94
San Juan La Laguna,86,103,115,118,0,0,115,115,105,96,104,112
San Lucas Toliman,65,76,90,85,0,0,94,80,81,80,90,84
San Marcos La Laguna,87,95,102,108,0,0,113,109,102,90,101,102
San Pablo La Laguna,65,78,94,101,0,0,107,91,82,80,89,91
San Pedro La Laguna,110,114,131,140,0,0,136,132,128,120,121,133
Santa Catarina Palopo,77,85,100,96,0,0,106,94,94,84,97,96
Santa Cruz La Laguna,43,57,66,72,0,0,74,62,56,49,68,67


La tabla permite distinguir un mes ausente para todo el lago de un mes presente con pocas filas de un pais. La falta de una combinacion pais-mes-segmento no se interpreta automaticamente como cero visitantes, porque no se cuenta con un contrato que garantice una tabla completa de todas las combinaciones.

### **3.6. Coherencia entre visitantes, dias-persona y permanencia**

Se compara la permanencia registrada con el cociente entre dias-persona y visitantes, solo cuando el denominador es positivo. Tambien se revisan los casos de cero visitantes y las etiquetas de permanencia. Las diferencias numericas se consideran relevantes cuando superan una tolerancia de 0.00000001.

In [16]:
visitantes = numericos["visitantes"]
dias_persona = numericos["dias_persona"]
permanencia = numericos["permanencia_prom"]
tolerancia = 1e-8
cociente = dias_persona.div(visitantes.where(visitantes.gt(0)))
comparables = visitantes.gt(0) & dias_persona.notna() & permanencia.notna()
menor_dia = datos["es_consecutivo"].eq("menor a 1 dia")
mayor_dia = datos["es_consecutivo"].eq("mayor a 1 dia")

reglas = {
    "Permanencia no coincide con dias-persona / visitantes": comparables & permanencia.sub(cociente).abs().gt(tolerancia),
    "Visitantes positivos sin permanencia": visitantes.gt(0) & faltantes["permanencia_prom"],
    "Cero visitantes con dias-persona positivos": visitantes.eq(0) & dias_persona.gt(0),
    "Dias-persona menores que visitantes": visitantes.notna() & dias_persona.notna() & dias_persona.lt(visitantes - tolerancia),
    "Excursionistas con dias-persona diferentes de visitantes": menor_dia & dias_persona.sub(visitantes).abs().gt(tolerancia),
    "Permanencia menor que un dia con visitantes positivos": visitantes.gt(0) & permanencia.lt(1 - tolerancia),
    "Permanencia superior a dias del mes": permanencia.gt(fechas.dt.days_in_month + tolerancia),
    "Etiqueta de permanencia no reconocida": ~(menor_dia | mayor_dia),
    "Mas de un dia con permanencia promedio igual a uno": mayor_dia & permanencia.sub(1).abs().le(tolerancia),
}
revision_coherencia = pd.DataFrame([
    {"revision": nombre, "registros": int(mascara.fillna(False).sum())}
    for nombre, mascara in reglas.items()
])
revision_coherencia

,revision,registros
0,Permanencia no coincide con dias-persona / vis...,0
1,Visitantes positivos sin permanencia,0
2,Cero visitantes con dias-persona positivos,0
3,Dias-persona menores que visitantes,0
4,Excursionistas con dias-persona diferentes de ...,0
5,Permanencia menor que un dia con visitantes po...,0
6,Permanencia superior a dias del mes,0
7,Etiqueta de permanencia no reconocida,0
8,Mas de un dia con permanencia promedio igual a...,778


In [17]:
sin_denominador = visitantes.eq(0) & dias_persona.eq(0) & faltantes["permanencia_prom"]
print("Permanencias vacias explicadas por cero visitantes y cero dias-persona:", int(sin_denominador.sum()))
print("Total de permanencias faltantes:", int(faltantes["permanencia_prom"].sum()))
casos_media_uno = ejemplos(reglas["Mas de un dia con permanencia promedio igual a uno"], columnas_contexto, limite=10)
casos_media_uno

Permanencias vacias explicadas por cero visitantes y cero dias-persona: 13
Total de permanencias faltantes: 13


,fila_excel,destino,fecha,pais,es_consecutivo,visitantes,dias_persona,permanencia_prom
67,69,Panajachel,2025-09-01 00:00:00,Luxemburgo,mayor a 1 dia,1,1,1
124,126,Panajachel,2025-10-01 00:00:00,Bahamas,mayor a 1 dia,1,1,1
226,228,Panajachel,2025-10-01 00:00:00,Trinidad Y Tobago,mayor a 1 dia,1,1,1
257,259,Panajachel,2025-11-01 00:00:00,Chipre,mayor a 1 dia,1,1,1
309,311,Panajachel,2025-11-01 00:00:00,Jamaica,mayor a 1 dia,1,1,1
312,314,Panajachel,2025-11-01 00:00:00,Kazajistan,mayor a 1 dia,1,1,1
349,351,Panajachel,2025-11-01 00:00:00,Singapur,mayor a 1 dia,1,1,1
362,364,Panajachel,2025-11-01 00:00:00,Ucrania,mayor a 1 dia,1,1,1
407,409,Panajachel,2025-12-01 00:00:00,Eslovaquia,mayor a 1 dia,2,2,1
416,418,Panajachel,2025-12-01 00:00:00,Filipinas,mayor a 1 dia,1,1,1


Los casos de cero visitantes y cero dias-persona no tienen un promedio definido: la division seria cero entre cero. Por eso una permanencia vacia en esas filas no debe rellenarse automaticamente con cero o uno. Por otro lado, una media igual a uno bajo la etiqueta mayor a un dia se registra como una tension entre la etiqueta y la medida, no como un error que se deba corregir automaticamente.

### **3.7. Distribucion y valores de mayor magnitud**

Se revisan las distribuciones separadas por grupo de permanencia y los registros con mas visitantes. Un valor grande no se considera incorrecto por su tamano: los municipios y mercados tienen escalas diferentes. No se eliminan observaciones mediante un umbral general de valores extremos.

In [18]:
distribuciones = (
    numericos[["visitantes", "dias_persona", "permanencia_prom"]]
    .assign(grupo=datos["es_consecutivo"])
    .groupby("grupo").describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .stack(level=0, future_stack=True)
)
distribuciones

count         mean  ...          95%            max
grupo                                                ...                            
mayor a 1 dia visitantes        4531.0   716.702913  ...   487.575835   94074.927645
              dias_persona      4531.0  4066.416524  ...  1107.500000  498014.261987
              permanencia_prom  4531.0     2.243752  ...     4.662958      30.000000
menor a 1 dia visitantes        5329.0  1260.887632  ...   846.800000  152399.452194
              dias_persona      5329.0  1260.887632  ...   846.800000  152399.452194
              permanencia_prom  5316.0     1.000000  ...     1.000000       1.000000

[6 rows x 9 columns]

In [19]:
indices_mayores = visitantes.nlargest(10).index
mayores_registros = datos.loc[indices_mayores, columnas_contexto].copy()
mayores_registros.insert(0, "fila_excel", mayores_registros.index + 2)
mayores_registros

,fila_excel,destino,fecha,pais,es_consecutivo,visitantes,dias_persona,permanencia_prom
424,426,Panajachel,2025-12-01 00:00:00,Guatemala,menor a 1 dia,152399.45219398304,152399.45219398304,1
704,706,Panajachel,2026-04-01 00:00:00,Guatemala,menor a 1 dia,140265.51024177033,140265.51024177033,1
292,294,Panajachel,2025-11-01 00:00:00,Guatemala,menor a 1 dia,131630.63507776224,131630.63507776224,1
975,977,Panajachel,2026-06-01 00:00:00,Guatemala,menor a 1 dia,128538.29824170365,128538.29824170365,1
1225,1227,Panajachel,2026-08-01 00:00:00,Guatemala,menor a 1 dia,128238.5242746029,128238.5242746029,1
47,49,Panajachel,2025-09-01 00:00:00,Guatemala,menor a 1 dia,126544.06621560802,126544.06621560802,1
841,843,Panajachel,2026-05-01 00:00:00,Guatemala,menor a 1 dia,125185.4386858043,125185.4386858043,1
563,565,Panajachel,2026-03-01 00:00:00,Guatemala,menor a 1 dia,123869.47465864316,123869.47465864316,1
6343,6345,San Pedro La Laguna,2025-12-01 00:00:00,Guatemala,menor a 1 dia,122626.98535356027,122626.98535356027,1
1101,1103,Panajachel,2026-07-01 00:00:00,Guatemala,menor a 1 dia,119880.6450423766,119880.6450423766,1


### **3.8. Resumen de hallazgos para discutir**

Las tablas anteriores dejan evidencia de las revisiones realizadas. El siguiente resumen separa los hechos observados de las posibles decisiones de preprocesamiento. Las propuestas no se han aplicado.

In [20]:
resumen_calidad = pd.DataFrame([
    {"hallazgo": "Pais no identificado", "evidencia": f"{int(vacios['pais'].sum())} vacios y {int(marcadores['pais'].sum())} marcadores de texto", "propuesta_para_discutir": "Unificar como faltante; conservar filas y no asignar pais por suposicion."},
    {"hallazgo": "Permanencia no definida", "evidencia": f"{int(sin_denominador.sum())} casos con cero visitantes y cero dias-persona", "propuesta_para_discutir": "Conservar como nulo por denominador cero, sin imputar."},
    {"hallazgo": "Etiqueta mayor a un dia con promedio uno", "evidencia": f"{int(reglas['Mas de un dia con permanencia promedio igual a uno'].sum())} registros", "propuesta_para_discutir": "Conservar etiqueta y valor; no reclasificar automaticamente."},
    {"hallazgo": "Valores decimales de visitantes", "evidencia": f"{int(con_decimales.sum())} registros en {len(decimales_por_pais)} paises", "propuesta_para_discutir": "Conservar precision; redondear solo resultados de presentacion."},
    {"hallazgo": "Meses sin cobertura", "evidencia": ", ".join(meses_ausentes), "propuesta_para_discutir": "Mostrar sin dato; reservar la estimacion para la fase de modelado."},
    {"hallazgo": "Columna sub_grupo constante", "evidencia": str(datos['sub_grupo'].unique().tolist()), "propuesta_para_discutir": "Evaluar retirarla de la tabla de analisis de Atitlan."},
    {"hallazgo": "Permanencia derivable", "evidencia": f"{int(reglas['Permanencia no coincide con dias-persona / visitantes'].sum())} discrepancias en cocientes comparables", "propuesta_para_discutir": "Evaluar recalcularla; al agrupar, usar cociente de sumas del segmento de pernocta."},
    {"hallazgo": "Dimension Excel incorrecta", "evidencia": f"Declarada {dimension_declarada}; reales {filas_fisicas} filas con cabecera", "propuesta_para_discutir": "Conservar una lectura que ignore esa dimension y validar la integridad de la carga."},
])
assert datos.equals(datos_originales), "El diagnostico modifico el DataFrame original."
print("Control final: los datos originales no fueron modificados.")
resumen_calidad

Control final: los datos originales no fueron modificados.


,hallazgo,evidencia,propuesta_para_discutir
0,Pais no identificado,13 vacios y 12 marcadores de texto,Unificar como faltante; conservar filas y no a...
1,Permanencia no definida,13 casos con cero visitantes y cero dias-persona,"Conservar como nulo por denominador cero, sin ..."
2,Etiqueta mayor a un dia con promedio uno,778 registros,Conservar etiqueta y valor; no reclasificar au...
3,Valores decimales de visitantes,998 registros en 5 paises,Conservar precision; redondear solo resultados...
4,Meses sin cobertura,"2026-01, 2026-02",Mostrar sin dato; reservar la estimacion para ...
5,Columna sub_grupo constante,['Lago Atitlan'],Evaluar retirarla de la tabla de analisis de A...
6,Permanencia derivable,0 discrepancias en cocientes comparables,"Evaluar recalcularla; al agrupar, usar cocient..."
7,Dimension Excel incorrecta,Declarada A1:A1; reales 9861 filas con cabecera,Conservar una lectura que ignore esa dimension...


## **4. Criterios de preprocesamiento**

A partir de la revision de calidad, se establecen los siguientes acuerdos para preparar la hoja `Atitlan` del archivo `BD_moviidad_mensual.xlsx`. Esta seccion documenta lo que se implementara posteriormente en el pipeline, sin aplicar todavia las transformaciones ni modificar el archivo original.

### **4.1. Valores faltantes y presentacion del pais**

- En `pais`, se unificaran las 13 celdas vacias y los 12 valores escritos como `Null` bajo un valor nulo real. Se conservaran los 25 registros, porque siguen aportando informacion para los analisis que no requieren conocer el pais.
- Al mostrar los resultados, estos registros se agruparan bajo la etiqueta **Pais no identificado**, para que la ausencia de informacion sea visible. Esta etiqueta se utilizara en la presentacion y no se guardara como si fuera un pais real. Tampoco se asignaran estos registros automaticamente al grupo nacional o extranjero.
- No se imputara un pais ni se eliminaran filas por tener ese dato ausente. Este faltante se documentara como informacion no identificada en la fuente.
- Los 13 casos con cero visitantes y cero dias-persona tienen una permanencia promedio no definida. Cuando se recalcule esta medida, el resultado se mantendra nulo por denominador cero, sin sustituirlo por cero o uno.

### **4.2. Columnas y tipos de datos**

- Se retirara `sub_grupo` del dataset procesado de Atitlan, porque tiene el mismo valor en todos los registros. El alcance territorial seguira documentado como Lago de Atitlan.
- Se retirara `permanencia_prom`, porque puede obtenerse dividiendo `dias_persona` entre `visitantes`. Para calcular la estadia de una agrupacion, se utilizara la suma de dias-persona dividida entre la suma de visitantes del grupo mayor a un dia; no se promediaran promedios.
- Se convertiran `anio` y `mes` a enteros, `fecha` a fecha y hora, y `visitantes` y `dias_persona` a valores numericos con decimales. Las demas variables conservaran su significado categorico y se verificara que los nombres de columnas del dataset procesado sigan la convencion del proyecto.
- Los valores originales de visitantes y dias-persona se conservaran con su precision disponible. La aproximacion al entero superior acordada para visitantes se aplicara solamente al resultado final que se presente, despues de realizar los calculos.

### **4.3. Registros que se conservaran**

- Los 778 registros clasificados como mayor a un dia que tienen permanencia promedio igual a uno conservaran su etiqueta y sus cantidades originales. La igualdad no es suficiente para justificar una reclasificacion automatica.
- La guia de uso, seccion 5.1, pagina 4, indica que a las visitas de menos de un dia se les asigna permanencia uno por definicion. Esa es la regla documentada; no implica que su duracion real fuera de un dia completo.
- Para los casos del grupo mayor a un dia con cociente igual a uno, el archivo no documenta el motivo especifico. Un posible redondeo de valores cercanos a uno se registra solamente como hipotesis no verificada, no como explicacion confirmada ni como criterio para cambiar etiquetas. Tampoco se atribuyen al redondeo los unos del grupo menor a un dia, porque para ellos existe la regla explicita anterior.
- Se conservaran los registros con cero visitantes y cero dias-persona. Esos ceros son valores presentes en la fuente y no equivalen a los meses sin cobertura.
- No se eliminaran valores grandes solo por su magnitud. Tampoco se deduplicara sin evidencia: la revision actual no encontro filas ni claves repetidas.

### **4.4. Cobertura y ejecucion del pipeline**

- Enero y febrero de 2026 permaneceran como meses sin datos observados. La limpieza no agregara filas con cero visitantes ni imputara esos meses. Su estimacion corresponde a la etapa posterior de modelado y se diferenciara de los datos observados.
- La lectura de Excel debera recuperar toda la hoja aunque la dimension declarada sea incorrecta. Se validaran las columnas, los diez municipios, las claves y la coherencia entre fecha, anio y mes antes y despues de las transformaciones.
- Las transformaciones se implementaran en scripts por etapas dentro de `src/`, con rutas y contratos en `config.py` y utilidades compartidas en `utils.py`. Las salidas se guardaran en `data/processed/`; los archivos de `data/raw/` permaneceran intactos.
- El diccionario del dataset procesado documentara las columnas finales, las reglas aplicadas y la diferencia entre un dato ausente en la fuente y un valor derivado no definido. El notebook conservara la evidencia que respalda estas decisiones.
